# Dataset Inspection and Artifact Audit

Part of the research repository **Exposing Dataset Artifacts in Medical AI: When Fundus Models Learn Borders Instead of Disease**.

> **Reproducibility note:** dataset files are not distributed with this repository. The original experiment was developed in Google Colab and uses Drive paths under `Fundus_Artifact_Project`. Update those paths for your environment before execution.


In [ ]:
# Step 1: Mount Google Drive and configure dataset paths
import os
import shutil
from google.colab import drive
if os.path.ismount('/content/drive'):
    drive.flush_and_unmount()
    shutil.rmtree('/content/drive')
drive.mount('/content/drive', force_remount=True)
BASE_PATH = '/content/drive/MyDrive/Fundus_Artifact_Project'
APTOS_PATH = os.path.join(BASE_PATH, 'APTOS_2019')
MESSIDOR_PATH = os.path.join(BASE_PATH, 'Messidor_2')
APTOS_IMG_DIR = os.path.join(APTOS_PATH, 'train_images')
APTOS_CSV = os.path.join(APTOS_PATH, 'train.csv')
MESSIDOR_IMG_DIR = os.path.join(MESSIDOR_PATH, 'my_preprocessed')
MESSIDOR_CSV = os.path.join(MESSIDOR_PATH, 'grades.csv')

In [ ]:
# Step 2: Load metadata
import pandas as pd
aptos_df = pd.read_csv(APTOS_CSV)
messidor_df = pd.read_csv(MESSIDOR_CSV)
print('APTOS preview:')
print(aptos_df.head())
print('\nMessidor preview:')
print(messidor_df.head())

In [ ]:
# Step 3: Verify APTOS image availability
from glob import glob
aptos_imgs = set(os.path.basename(p) for p in glob(f'{APTOS_IMG_DIR}/*.png'))
aptos_df['exists'] = aptos_df['id_code'].apply(lambda x: f'{x}.png' in aptos_imgs)
missing_aptos = aptos_df[~aptos_df['exists']]
print(f'Missing APTOS images: {len(missing_aptos)}')
aptos_df = aptos_df[aptos_df['exists']]

In [ ]:
# Step 4: Visualize sample APTOS images
import matplotlib.pyplot as plt
import cv2
import numpy as np
def show_samples(df, img_dir, filename_col, label_col, n=6, prefix='aptos'):
    sample = df.sample(n)
    fig, axs = plt.subplots(1, n, figsize=(16, 4))
    for i, (_, row) in enumerate(sample.iterrows()):
        img_path = os.path.join(img_dir, f"{row[filename_col]}.png")
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        axs[i].imshow(img); axs[i].axis('off'); axs[i].set_title(f"Label: {row[label_col]}")
        save_path = os.path.join(BASE_PATH, 'Results', f'{prefix}_sample_{i+1}.png')
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    plt.tight_layout(); plt.show()
aptos_df['id_code'] = aptos_df['id_code'].astype(str)
show_samples(aptos_df, APTOS_IMG_DIR, 'id_code', 'diagnosis')

In [ ]:
# Step 5: Visualize sample Messidor-2 images
def show_samples_safe(df, img_dir, filename_col, label_col, n=6, prefix='messidor'):
    sample = df.sample(n); fig, axs = plt.subplots(1, n, figsize=(16, 4)); count = 0
    for _, row in sample.iterrows():
        base_name = row[filename_col].split('.')[0]
        found = glob(os.path.join(img_dir, f'{base_name}.*'))
        if not found: continue
        img = cv2.imread(found[0])
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axs[count].imshow(img); axs[count].axis('off'); axs[count].set_title(f"Label: {row[label_col]}")
        save_path = os.path.join(BASE_PATH, 'Results', f'{prefix}_sample_{count+1}.png')
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR)); count += 1
        if count >= n: break
    plt.tight_layout(); plt.show()
show_samples_safe(messidor_df, MESSIDOR_IMG_DIR, 'filename', 'grade')

In [ ]:
# Step 6: Inspect APTOS image-resolution statistics
dims = []
for img_name in aptos_df['id_code'].sample(30):
    img = cv2.imread(os.path.join(APTOS_IMG_DIR, f'{img_name}.png'))
    dims.append(img.shape[:2])
dims = np.array(dims)
print('Average resolution:', dims.mean(axis=0))
print('Maximum resolution:', dims.max(axis=0))
print('Minimum resolution:', dims.min(axis=0))